In [19]:
import pandas as pd

#appointments = pd.read_csv("appointments.csv")
#patients = pd.read_csv("patients.csv")
#treatments = pd.read_csv("treatments.csv")
#doctors = pd.read_csv("doctors.csv")
#billing = pd.read_csv("billing.csv")

df = pd.read_csv("KaggleV2-May-2016.csv")
print(df.columns.tolist())
print(df.shape)
print(df.head(3))

print(appointments.head())

['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']
(110527, 14)
      PatientId  AppointmentID Gender          ScheduledDay  \
0  2.987250e+13        5642903      F  2016-04-29T18:38:08Z   
1  5.589978e+14        5642503      M  2016-04-29T16:08:27Z   
2  4.262962e+12        5642549      F  2016-04-29T16:19:04Z   

         AppointmentDay  Age    Neighbourhood  Scholarship  Hipertension  \
0  2016-04-29T00:00:00Z   62  JARDIM DA PENHA            0             1   
1  2016-04-29T00:00:00Z   56  JARDIM DA PENHA            0             0   
2  2016-04-29T00:00:00Z   62    MATA DA PRAIA            0             0   

   Diabetes  Alcoholism  Handcap  SMS_received No-show  
0         0           0        0             0      No  
1         0           0        0             0      No  
2         0           0        0             0      No  
  appoi

In [22]:
print(monthly["total_appointments"].describe())
print("\nSample facilities:", monthly["hospital_branch"].unique()[:5].tolist())

count    35.000000
mean      5.714286
std       2.652096
min       2.000000
25%       4.000000
50%       5.000000
75%       7.000000
max      13.000000
Name: total_appointments, dtype: float64

Sample facilities: ['Central Hospital', 'Eastside Clinic', 'Westside Clinic']


In [26]:
#Dataset: Medical Appointment No-Shows (Kaggle)
#          kaggle.com/joniarroba/noshowappointments
#          110,527 appointments | Brazil, 2016

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")


# 1. LOAD & INSPECT
print("1. LOAD & INSPECT")

df = pd.read_csv("KaggleV2-May-2016.csv")
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nHead:\n{df.head(3)}")


# 2. CLEAN & PREPARE
print("2. CLEAN & PREPARE")

# Parse dates
df["ScheduledDay"]   = pd.to_datetime(df["ScheduledDay"]).dt.tz_localize(None)
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"]).dt.tz_localize(None)

# Binary target: 1 = no-show, 0 = showed up
df["no_show"] = (df["No-show"] == "Yes").astype(int)
print(f"\nTarget distribution:\n{df['no_show'].value_counts()}")
print(f"No-show rate: {df['no_show'].mean():.1%}")

# Drop known bad rows (age errors documented in this dataset)
before = len(df)
df = df[(df["Age"] >= 0) & (df["Age"] <= 110)]
print(f"\nDropped {before - len(df)} rows with invalid age. Remaining: {len(df):,}")

# Drop rows where appointment was scheduled AFTER the appointment day
# (data entry errors — negative wait time is nonsensical)
df = df[df["AppointmentDay"] >= df["ScheduledDay"].dt.normalize()]
print(f"After removing negative wait times: {len(df):,} rows")


# 3. FEATURE ENGINEERING
print("3. FEATURE ENGINEERING")

# Feature 1: Wait time
# Days between when the appointment was scheduled and when it actually occurs.
# Longer waits are strongly associated with higher no-show rates in the
# literature (Dantas et al., 2018).
df["wait_days"] = (
    df["AppointmentDay"] - df["ScheduledDay"].dt.normalize()
).dt.days
print(f"\nWait days summary:\n{df['wait_days'].describe().round(1)}")

# Feature 2: Appointment day of week
# Attendance patterns vary by day, patients may be more likely to skip
# Friday or Monday appointments than midweek ones.
df["appt_day_of_week"] = df["AppointmentDay"].dt.dayofweek  # 0=Mon, 6=Sun
dow_labels = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
df["appt_day_name"] = df["appt_day_of_week"].map(dow_labels)
print(f"\nAppointments by day of week:\n{df['appt_day_name'].value_counts()}")

# Feature 3: Prior no-show rate per patient
# PatientId repeats across appointments. A patient's historical no-show
# behavior is one of the strongest predictors of future no-shows.
#
# IMPORTANT — no data leakage: we sort by ScheduledDay and use an expanding
# window shifted by 1, so each row only sees appointments that came before it.
# A patient's first appointment is filled with the global no-show rate.

df = df.sort_values(["PatientId", "ScheduledDay"]).reset_index(drop=True)

df["_cumulative_noshows"] = (
    df.groupby("PatientId")["no_show"]
    .transform(lambda x: x.shift(1).expanding().sum())
)
df["_cumulative_appts"] = (
    df.groupby("PatientId")["no_show"]
    .transform(lambda x: x.shift(1).expanding().count())
)

global_nsr = df["no_show"].mean()
df["prior_noshow_rate"] = (
    df["_cumulative_noshows"] / df["_cumulative_appts"]
).fillna(global_nsr)

df.drop(columns=["_cumulative_noshows", "_cumulative_appts"], inplace=True)

print(f"\nPrior no-show rate summary:\n{df['prior_noshow_rate'].describe().round(3)}")
print(f"(First-visit rows filled with global rate: {global_nsr:.3f})")
print(f"\nFinal dataset: {df.shape[0]:,} rows | "
      f"Engineered features: wait_days, appt_day_of_week, prior_noshow_rate")


# 4. EDA
print("4. EDA")

# 4a. No-show rate by wait time bucket
df["wait_bucket"] = pd.cut(
    df["wait_days"],
    bins=[-1, 0, 7, 30, 60, 999],
    labels=["Same day", "1-7 days", "8-30 days", "31-60 days", "60+ days"]
)
wait_nsr = df.groupby("wait_bucket", observed=True)["no_show"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
wait_nsr.plot(kind="bar", ax=ax, edgecolor="black", color="steelblue")
ax.set_title("No-Show Rate by Wait Time")
ax.set_xlabel("Days Between Scheduling and Appointment")
ax.set_ylabel("No-Show Rate")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig("eda_noshow_by_wait.png", dpi=150)
plt.close()
print("Saved: eda_noshow_by_wait.png")

# 4b. No-show rate by day of week
dow_nsr = (
    df.groupby("appt_day_name", observed=True)["no_show"]
    .mean()
    .reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"])
)

fig, ax = plt.subplots(figsize=(8, 4))
dow_nsr.plot(kind="bar", ax=ax, edgecolor="black", color="steelblue")
ax.set_title("No-Show Rate by Day of Week")
ax.set_xlabel("Appointment Day")
ax.set_ylabel("No-Show Rate")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig("eda_noshow_by_dow.png", dpi=150)
plt.close()
print("Saved: eda_noshow_by_dow.png")

# 4c. No-show rate by SMS and age
sms_nsr = df.groupby("SMS_received")["no_show"].mean()
print(f"\nNo-show rate by SMS:\n{sms_nsr.rename({0:'No SMS', 1:'SMS Sent'}).round(3)}")

df["age_bucket"] = pd.cut(
    df["Age"],
    bins=[0, 17, 34, 54, 74, 110],
    labels=["0-17", "18-34", "35-54", "55-74", "75+"]
)
age_nsr = df.groupby("age_bucket", observed=True)["no_show"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

age_nsr.plot(kind="bar", ax=axes[0], edgecolor="black", color="steelblue")
axes[0].set_title("No-Show Rate by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("No-Show Rate")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# Prior no-show rate distribution split by outcome
for label, grp in df.groupby("no_show"):
    grp["prior_noshow_rate"].plot(
        kind="kde", ax=axes[1],
        label="No-Show" if label == 1 else "Showed Up"
    )
axes[1].set_title("Prior No-Show Rate Distribution by Outcome")
axes[1].set_xlabel("Prior No-Show Rate")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_age_and_prior.png", dpi=150)
plt.close()
print("Saved: eda_age_and_prior.png")

# 4d. Correlation heatmap
heatmap_cols = [
    "no_show", "wait_days", "appt_day_of_week", "prior_noshow_rate",
    "Age", "SMS_received", "Scholarship",
    "Hipertension", "Diabetes", "Alcoholism", "Handcap"
]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[heatmap_cols].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=150)
plt.close()
print("Saved: eda_correlation.png")


# 5. BASELINE LOGISTIC REGRESSION
print("\n" + "=" * 60)
print("5. BASELINE LOGISTIC REGRESSION")
print("=" * 60)

feature_cols = [
    # Engineered features
    "wait_days",
    "appt_day_of_week",
    "prior_noshow_rate",
    # Demographics
    "Age",
    "Gender",
    # Operational
    "SMS_received",
    # Health / socioeconomic
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
]

model_df = df[feature_cols + ["no_show"]].copy()
model_df["Gender"] = (model_df["Gender"] == "M").astype(int)
model_df = model_df.dropna()
print(f"Modeling dataset: {model_df.shape[0]:,} rows × {len(feature_cols)} features")

X = model_df[feature_cols]
y = model_df["no_show"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Fit
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# Evaluate
y_pred    = lr.predict(X_test_s)
y_prob    = lr.predict_proba(X_test_s)[:, 1]
roc_auc   = roc_auc_score(y_test, y_prob)
cv_scores = cross_val_score(lr, scaler.fit_transform(X), y, cv=5, scoring="roc_auc")

print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
print(f"ROC-AUC (test):      {roc_auc:.4f}")
print(f"ROC-AUC (5-fold CV): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Coefficients
coef_df = pd.DataFrame({
    "feature":     feature_cols,
    "coefficient": lr.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print(f"\nCoefficients:\n{coef_df.to_string(index=False)}")

# Plots
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

ax1 = fig.add_subplot(gs[0])
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["Showed Up", "No-Show"]
).plot(ax=ax1, colorbar=False)
ax1.set_title("Confusion Matrix")

ax2 = fig.add_subplot(gs[1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
ax2.plot([0, 1], [0, 1], "k--", lw=1)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve – Logistic Regression")
ax2.legend()

ax3 = fig.add_subplot(gs[2])
colors = ["steelblue" if c > 0 else "salmon" for c in coef_df["coefficient"]]
ax3.barh(coef_df["feature"], coef_df["coefficient"], color=colors, edgecolor="black")
ax3.axvline(0, color="black", linewidth=0.8)
ax3.set_title("Feature Coefficients")
ax3.set_xlabel("Coefficient Value")

plt.tight_layout()
plt.savefig("model_logistic_regression.png", dpi=150)
plt.close()
print("\nSaved: model_logistic_regression.png")

print("PIPELINE COMPLETE")
print("\nOutputs:")
print("  eda_noshow_by_wait.png        – no-show rate by wait time bucket")
print("  eda_noshow_by_dow.png         – no-show rate by day of week")
print("  eda_age_and_prior.png         – age breakdown + prior rate distribution")
print("  eda_correlation.png           – feature correlation heatmap")
print("  model_logistic_regression.png – confusion matrix, ROC, coefficients")

1. LOAD & INSPECT
Shape: (110527, 14)

Dtypes:
PatientId         float64
AppointmentID       int64
Gender             object
ScheduledDay       object
AppointmentDay     object
Age                 int64
Neighbourhood      object
Scholarship         int64
Hipertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show            object
dtype: object

Missing values:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

Head:
      PatientId  AppointmentID Gender          ScheduledDay  \
0  2.987250e+13        5642903      F  2016-04-29T18:38:08Z   
1  5.589978e+14        5642503      M  2016-04-29T16:08:27Z   
2  4.262962e+12        5642549      F  2016-04-29T16:19:04Z   

     